In [103]:
import pandas as pd

df = pd.read_excel('./GSAF5.xls')

print("Información sobre el DataFrame:")
df.info()

Información sobre el DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 7125 entries, 0 to 7124
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            7125 non-null   object 
 1   Year            7123 non-null   float64
 2   Type            7107 non-null   str    
 3   Country         7075 non-null   str    
 4   State           6638 non-null   str    
 5   Location        6558 non-null   str    
 6   Activity        6542 non-null   str    
 7   Name            6907 non-null   str    
 8   Sex             6547 non-null   str    
 9   Age             4131 non-null   object 
 10  Injury          7089 non-null   str    
 11  Fatal Y/N       6564 non-null   object 
 12  Time            3598 non-null   object 
 13  Species         3994 non-null   str    
 14  Source          7105 non-null   object 
 15  pdf             6799 non-null   object 
 16  href formula    6794 non-null   str    
 17  href        

In [104]:
def imprimir_nulos_columna(df, columna):
  
    if df is None:
        raise ValueError("df es None: una celda anterior usó 'df = ...' con inplace=True "
                         "o una función no devuelve df.")
    print(f"""Número de nulos en la columna '{columna}':""")
    print(df[columna].isnull().sum())

def imprimir_nulos(df):
    print(df.isnull().sum())

def limpiar_columna_texto(df, columna, valor_nulo='unknown'):
    """Limpia una columna de texto: espacios, mayúsculas/minúsculas, vacíos y nulos."""
    df = df.copy()  # no modifica el DataFrame original
    df[columna] = (
        df[columna]
        .str.strip()                           # quita espacios al inicio y al final
        .str.replace(r'\s+', ' ', regex=True)  # varios espacios -> uno
        .str.lower()                           # "Surfing" y "surfing" pasan a ser lo mismo
        .replace('', valor_nulo)               # cadenas vacías -> valor común
        .fillna(valor_nulo)                    # nulos -> valor común
    )
    return df


def comprobar_columna(df_antes, df_despues, columna, top=30):
    """Muestra cómo ha cambiado una columna tras la limpieza."""
    print(f"Filas antes / después: {df_antes.shape[0]} / {df_despues.shape[0]}")
    print(f"Nulos antes / después: {df_antes[columna].isnull().sum()} / {df_despues[columna].isnull().sum()}")
    print(f"Valores distintos antes / después: {df_antes[columna].nunique()} / {df_despues[columna].nunique()}")
    print(f"\nTop {top} de '{columna}':")
    print(df_despues[columna].value_counts().head(top))

def limpiar_columna_numerica(df, columna, minimo=None, maximo=None, entero=False, relleno=None):

    df = df.copy()  # no modifica el DataFrame original
    serie = df[columna]

    if serie.dtype == 'object':
        # extrae el primer número del texto ("20s" -> 20, "teen" -> NaN)
        serie = serie.astype(str).str.extract(r'(-?\d+\.?\d*)')[0]

    serie = pd.to_numeric(serie, errors='coerce')  # lo que no sea número -> NaN

    if minimo is not None:
        serie = serie.where(serie >= minimo)       # por debajo del mínimo -> NaN
    if maximo is not None:
        serie = serie.where(serie <= maximo)       # por encima del máximo -> NaN

    if relleno == 'mediana':
        serie = serie.fillna(serie.median())
    elif relleno == 'media':
        serie = serie.fillna(serie.mean())
    elif relleno is not None:
        serie = serie.fillna(relleno)

    if entero:
        serie = serie.round().astype('Int64')      # entero que admite nulos

    df[columna] = serie
    return df


def comprobar_columna_numerica(df_antes, df_despues, columna):
    """Muestra cómo ha cambiado una columna numérica tras la limpieza."""
    print(f"Filas antes / después: {df_antes.shape[0]} / {df_despues.shape[0]}")
    print(f"Tipo antes / después: {df_antes[columna].dtype} / {df_despues[columna].dtype}")
    print(f"Nulos antes / después: {df_antes[columna].isnull().sum()} / {df_despues[columna].isnull().sum()}")
    print(f"\nEstadísticas de '{columna}':")
    print(df_despues[columna].describe())



In [105]:
# Limpieza de datos: Columna Activity
df_antes = df
df = limpiar_columna_texto(df, 'Activity')
comprobar_columna(df_antes, df, 'Activity')

Filas antes / después: 7125 / 7125
Nulos antes / después: 583 / 0
Valores distintos antes / después: 1617 / 1554

Top 30 de 'Activity':
Activity
surfing            1162
swimming           1075
unknown             588
fishing             514
spearfishing        405
wading              178
bathing             167
diving              155
snorkeling          138
standing            115
scuba diving        106
body boarding        70
boogie boarding      60
body surfing         55
kayaking             44
free diving          35
treading water       33
fell overboard       33
pearl diving         32
surf skiing          24
windsurfing          20
floating             18
canoeing             17
walking              17
kayak fishing        16
shark fishing        16
playing              14
surf fishing         14
sea disaster         14
rowing               14
Name: count, dtype: int64


In [106]:
# Limpieza de datos: Columna Country
df_antes = df
df = limpiar_columna_texto(df, 'Country')
comprobar_columna(df_antes, df, 'Country')

Filas antes / después: 7125 / 7125
Nulos antes / después: 50 / 0
Valores distintos antes / después: 257 / 215

Top 30 de 'Country':
Country
usa                 2597
australia           1538
south africa         599
bahamas              146
new zealand          146
papua new guinea     136
brazil               125
mexico               107
italy                 72
fiji                  71
new caledonia         68
philippines           65
reunion               60
egypt                 53
mozambique            52
unknown               50
cuba                  49
spain                 47
india                 41
french polynesia      40
japan                 36
croatia               35
jamaica               32
panama                32
solomon islands       30
iran                  29
indonesia             25
england               25
greece                24
hong kong             24
Name: count, dtype: int64


In [107]:
def limpiar_columna_categorica(df, columna, valores_validos, valor_desconocido='unknown', correcciones=None):

    df = df.copy()  # no modifica el DataFrame original
    serie = df[columna].str.strip().str.lower()   # quita espacios y unifica minúsculas
    if correcciones:
        serie = serie.replace(correcciones)       # corrige casos concretos
    df[columna] = serie.where(serie.isin(valores_validos), valor_desconocido)  # inválidos y nulos -> unknown
    
    return df

def comprobar_columna_categorica(df_antes, df_despues, columna):

    print(f"Filas antes / después: {df_antes.shape[0]} / {df_despues.shape[0]}")
    print(f"\nValores de '{columna}' ANTES:")
    print(df_antes[columna].value_counts(dropna=False).head(15))
    print(f"\nValores de '{columna}' DESPUÉS:")
    print(df_despues[columna].value_counts(dropna=False))



# Limpieza de datos: Columna Sex (solo m, f o unknown)
correcciones_sex = {
    'm x 2': 'm'
}

df_antes = df
df = limpiar_columna_categorica(df, 'Sex', valores_validos=['m', 'f'], correcciones=correcciones_sex)
comprobar_columna_categorica(df_antes, df, 'Sex')

Filas antes / después: 7125 / 7125

Valores de 'Sex' ANTES:
Sex
M        5709
F         818
NaN       578
M           8
?           2
F           2
N           2
M F         1
 M          1
m           1
lli         1
M x 2       1
.           1
Name: count, dtype: int64

Valores de 'Sex' DESPUÉS:
Sex
m          5720
f           820
unknown     585
Name: count, dtype: int64


In [108]:
df_antes = df
imprimir_nulos_columna(df_antes, 'Age')
print()
df = limpiar_columna_numerica(df, 'Age', minimo=1, maximo=100, entero=True)
comprobar_columna_numerica(df_antes, df, 'Age')
print()
imprimir_nulos_columna(df, 'Age')

Número de nulos en la columna 'Age':
2994

Filas antes / después: 7125 / 7125
Tipo antes / después: object / Int64
Nulos antes / después: 2994 / 3069

Estadísticas de 'Age':
count       4056.0
mean     28.199704
std      14.703248
min            1.0
25%           17.0
50%           24.0
75%           37.0
max           87.0
Name: Age, dtype: Float64

Número de nulos en la columna 'Age':
3069


In [109]:
df_antes = df
df = limpiar_columna_numerica(df, 'Year', entero=True)
comprobar_columna_numerica(df_antes, df, 'Year')

Filas antes / después: 7125 / 7125
Tipo antes / después: float64 / Int64
Nulos antes / después: 2 / 2

Estadísticas de 'Year':
count         7123.0
mean     1936.649586
std       269.808177
min              0.0
25%           1948.0
50%           1987.0
75%           2010.0
max           2026.0
Name: Year, dtype: Float64
